# BODMAS malware detection: SVM with feature selection (k grid search)

**Goal.** Detect whether a Windows executable is malware (1) or benign (0) using the 2,381 EMBER v2 features in BODMAS, and find out how many of those features an SVM really needs.

**What this notebook does**
1. Loads `bodmas.npz` and makes a train/test split.
2. Builds a pipeline: variance filter, scaling, `SelectKBest` feature selection, linear SVM.
3. Grid-searches the number of kept features `k` and the regularization strength `C` with cross-validation.
4. Refits on the full training set and compares three models on the held-out test set:
   - `LinearSVC` on all columns
   - `LinearSVC` on the best `k` columns
   - RBF-approximation SVM (Nystroem + linear SVM) on the best `k` columns

**Important design choice.** Scaling and feature selection live *inside* the pipeline, so they are re-fit on each cross-validation fold and on the training data only. This prevents test information from leaking into training. Because of this, the pipeline expects **unscaled** features.

The outputs shown below were produced by the author's run on the full dataset.

## 1. Imports

In [2]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

## 2. Settings

| Setting | Meaning |
|---|---|
| `NPZ_PATH` | Location of `bodmas.npz` (adjust to your path) |
| `TIME_SPLIT` | `False`: random stratified 80/20 split. `True`: train on the oldest 80% of samples, test on the newest 20% (a more realistic malware setup, needs the metadata CSV) |
| `K_GRID` | Numbers of features to try. `"all"` keeps every column that survives the variance filter |
| `C_GRID` | SVM regularization strengths. Smaller `C` means stronger regularization |
| `CV_FOLDS` | Cross-validation folds |
| `GRID_SAMPLE` | Rows used for the grid search. A subsample keeps it fast; `None` uses all training rows |
| `SCORING` | Metric used to choose `k` and `C` (F1 balances precision and recall) |

In [3]:
NPZ_PATH = "dataset/bodmas/bodmas.npz"            # adjust to your path
META_PATH = "dataset/bodmas/bodmas_metadata.csv"  # only used if TIME_SPLIT=True
TIME_SPLIT = False
K_GRID = [50, 100, 200, 300, 600, 1000, "all"]
C_GRID = [0.01, 0.1, 1.0]
CV_FOLDS = 3
GRID_SAMPLE = 30000
SCORING = "f1"
N_JOBS = -1
SEED = 42

## 3. Loading the data

The split is done on row *indices*, so `X` and `y` always stay aligned. With a random split the class ratio is preserved (`stratify=y`). With `TIME_SPLIT=True`, rows are ordered by first-seen timestamp from `bodmas_metadata.csv`.

In [4]:
def load_data():
    data = np.load(NPZ_PATH)
    X, y = data["X"], data["y"]

    if TIME_SPLIT:
        meta = pd.read_csv(META_PATH)
        ts = pd.to_datetime(meta["timestamp"], format="mixed", utc=True)
        order = np.argsort(ts.values, kind="stable")
        cut = int(0.8 * len(order))
        tr, te = order[:cut], order[cut:]
    else:
        tr, te = train_test_split(np.arange(len(y)), test_size=0.2,
                                  stratify=y, random_state=SEED)
    return X[tr], X[te], y[tr], y[te]

## 4. Pipelines

**`linear_pipe()`** has four steps:
1. `VarianceThreshold`: drops columns that are constant (zero variance). They carry no information.
2. `StandardScaler`: puts every feature on the same scale, which SVMs need.
3. `SelectKBest(f_classif)`: keeps the `k` columns whose values differ most between benign and malware (ANOVA F-test). `k` is set by the grid search.
4. `LinearSVC`: the classifier. `class_weight="balanced"` compensates for the benign/malware imbalance; `dual=False` is the faster solver when there are more samples than features.

**`rbf_pipe(k)`** uses the same first three steps, then approximates an RBF kernel with `Nystroem` (2,000 components) and trains a linear SVM (`SGDClassifier` with hinge loss) on the result. A true kernel SVM would be far too slow on about 107k samples, so this is the standard shortcut.

In [5]:
def linear_pipe():
    return Pipeline([
        ("var", VarianceThreshold(0.0)),
        ("scale", StandardScaler()),
        ("select", SelectKBest(f_classif, k="all")),
        ("svm", LinearSVC(dual=False, class_weight="balanced", max_iter=5000)),
    ])


def rbf_pipe(k, n_components=2000):
    return Pipeline([
        ("var", VarianceThreshold(0.0)),
        ("scale", StandardScaler()),
        ("select", SelectKBest(f_classif, k=k)),
        ("rbf", Nystroem(kernel="rbf", n_components=n_components, random_state=SEED)),
        ("svm", SGDClassifier(loss="hinge", class_weight="balanced",
                              alpha=1e-5, max_iter=50, tol=None, random_state=SEED)),
    ])

## 5. Evaluation helper

Fits a model, times the fit, and reports accuracy, precision, recall, F1, ROC-AUC, and the raw error counts:
- **false positives** (benign flagged as malware, "false alarms")
- **false negatives** (malware that slipped through, usually the costlier error)

In [6]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te, n_feats):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    fit_s = time.time() - t0
    pred = model.predict(X_te)
    score = model.decision_function(X_te)
    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    return {
        "model": name, "n_features": n_feats,
        "accuracy": accuracy_score(y_te, pred),
        "precision": precision_score(y_te, pred),
        "recall": recall_score(y_te, pred),
        "f1": f1_score(y_te, pred),
        "roc_auc": roc_auc_score(y_te, score),
        "false_pos": fp, "false_neg": fn,
        "fit_seconds": round(fit_s, 1),
    }

## 6. The comparison

`run_comparison` does three things:
1. **Grid search** over `k` and `C` (on a stratified subsample of the training set for speed) with stratified CV. It also picks the best `C` for the all-columns model, so that baseline is tuned fairly too.
2. **Plots** the CV score against `k` for each `C`.
3. **Refits** the three models on the *full* training set and scores them on the test set.

It also saves `svm_k_gridsearch.csv`, `svm_k_test_results.csv`, and `svm_k_cv_curve.png`.

In [7]:
def run_comparison(X_train, X_test, y_train, y_test, out_prefix="svm_k"):
    # --- 1. grid search over k and C (on a stratified subsample for speed) ---
    if GRID_SAMPLE and GRID_SAMPLE < len(y_train):
        Xg, _, yg, _ = train_test_split(X_train, y_train, train_size=GRID_SAMPLE,
                                        stratify=y_train, random_state=SEED)
    else:
        Xg, yg = X_train, y_train

    grid = GridSearchCV(
        linear_pipe(),
        {"select__k": K_GRID, "svm__C": C_GRID},
        cv=StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED),
        scoring=SCORING, n_jobs=N_JOBS, verbose=1, return_train_score=False,
    )
    t0 = time.time()
    grid.fit(Xg, yg)
    print(f"\nGrid search done in {time.time() - t0:.0f}s")
    print("Best params:", grid.best_params_, f"| CV {SCORING}: {grid.best_score_:.4f}")

    cv = pd.DataFrame(grid.cv_results_)
    cv["k"] = cv["param_select__k"].astype(str)
    cv["C"] = cv["param_svm__C"]
    cv_table = cv[["k", "C", "mean_test_score", "std_test_score", "mean_fit_time"]]
    cv_table = cv_table.sort_values("mean_test_score", ascending=False)
    cv_table.to_csv(f"{out_prefix}_gridsearch.csv", index=False)
    print(cv_table.head(10).to_string(index=False))

    best_k, best_C = grid.best_params_["select__k"], grid.best_params_["svm__C"]

    # best C for the all-features model (so the baseline is fairly tuned too)
    all_rows = cv[cv["k"] == "all"].sort_values("mean_test_score", ascending=False)
    all_C = all_rows.iloc[0]["C"]

    # --- 2. plot CV score vs k ---
    fig, ax = plt.subplots(figsize=(7, 4))
    n_cols = X_train.shape[1]
    for C in C_GRID:
        sub = cv[cv["C"] == C].copy()
        sub["k_num"] = sub["k"].map(lambda v: n_cols if v == "all" else int(v))
        sub = sub.sort_values("k_num")
        ax.plot(sub["k_num"], sub["mean_test_score"], marker="o", label=f"C={C}")
    ax.set_xscale("log"); ax.set_xlabel("k (number of selected features)")
    ax.set_ylabel(f"CV {SCORING}"); ax.set_title("LinearSVC: CV score vs. k")
    ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
    fig.savefig(f"{out_prefix}_cv_curve.png", dpi=150)
    plt.show()

    # --- 3. refit on the FULL training set and score on the test set ---
    results = []

    m_all = linear_pipe().set_params(select__k="all", svm__C=all_C)
    results.append(evaluate(f"LinearSVC, all columns (C={all_C})", m_all,
                            X_train, y_train, X_test, y_test,
                            "all (after variance filter)"))

    m_k = linear_pipe().set_params(select__k=best_k, svm__C=best_C)
    results.append(evaluate(f"LinearSVC, k={best_k} (C={best_C})", m_k,
                            X_train, y_train, X_test, y_test, best_k))

    m_rbf = rbf_pipe(best_k)
    results.append(evaluate(f"Nystroem-RBF SVM, k={best_k}", m_rbf,
                            X_train, y_train, X_test, y_test, best_k))

    res = pd.DataFrame(results)
    res.to_csv(f"{out_prefix}_test_results.csv", index=False)
    pd.set_option("display.width", 200)
    print("\n=== Test-set comparison ===")
    print(res.round(4).to_string(index=False))
    return grid, res

## 7. Run it

The full run took about 17 minutes for the grid search (63 fits) plus about 12 minutes for the final refits, almost all of it for the all-columns model.

The output below is the author's run: 107,548 training rows, 26,887 test rows, 2,381 features, with 42.6% of the training samples being malware.

In [ ]:
X_train, X_test, y_train, y_test = load_data()
print("train:", X_train.shape, "test:", X_test.shape,
      "| malware share (train):", round(float(y_train.mean()), 3))
grid, res = run_comparison(X_train, X_test, y_train, y_test)

train: (107548, 2381) test: (26887, 2381) | malware share (train): 0.426
Fitting 3 folds for each of 21 candidates, totalling 63 fits

Grid search done in 1031s
Best params: {'select__k': 1000, 'svm__C': 0.01} | CV f1: 0.9809
   k    C  mean_test_score  std_test_score  mean_fit_time
1000 0.01         0.980934        0.001870      20.210118
 all 0.01         0.979851        0.001792     145.392774
1000 0.10         0.978209        0.001496      63.426094
 600 0.10         0.977887        0.001605      21.662582
 600 0.01         0.977805        0.001005       8.383515
 600 1.00         0.976496        0.002126      74.266658
 all 0.10         0.976271        0.002793     268.720958
1000 1.00         0.976186        0.001276     169.501185
 all 1.00         0.973994        0.003191     343.875606
 300 1.00         0.969351        0.001947       8.834597

=== Test-set comparison ===
                          model                  n_features  accuracy  precision  recall     f1  roc_auc  f

## 8. Interpreting the results

### Grid search
- The best setting was **`k = 1000`, `C = 0.01`** with a CV F1 of 0.981. Smaller `C` (stronger regularization) scored better at every `k`, and `C = 1` was the slowest.
- The gap between `k = 1000` (0.9809) and all columns (0.9799) is smaller than the fold-to-fold spread (about ±0.002), so cross-validation cannot really tell them apart. `k = 300` (0.969) is clearly worse.
- Fit time grows sharply with more features and larger `C`: 20 s for the best setting versus 344 s for all columns with `C = 1`.

### Test-set comparison
Roughly 11,450 of the 26,887 test samples are malware and 15,440 are benign (from the 42.6% malware share).

| Model | F1 | Recall | Precision | Missed malware | False alarms | Fit time |
|---|---|---|---|---|---|---|
| LinearSVC, all columns | 0.9880 | 0.9921 | 0.9839 | 90 | 186 | 598 s |
| LinearSVC, k=1000 | 0.9856 | 0.9879 | 0.9833 | 139 | 192 | 41 s |
| Nystroem-RBF, k=1000 | 0.9784 | 0.9701 | 0.9869 | 343 | 147 | 49 s |

1. **All columns vs. k = 1000.** All columns is slightly better (F1 +0.0024, 49 fewer missed malware) but takes about 14 times longer to train. The two have almost the same false alarms, and k = 1000 has a marginally higher ROC-AUC (0.9980 vs 0.9972), meaning both rank samples about equally well. Feature selection costs very little accuracy here.
2. **The RBF model did not help.** It has the fewest false alarms but misses the most malware (343). This suggests the classes are already nearly linearly separable with these features. The RBF model was not tuned (`alpha`, `n_components`, and iterations were fixed), so this does not prove an RBF kernel cannot do better.
3. **Test scores match CV.** The test F1 (0.9856) is close to the CV F1 (0.981), which is consistent with the pipeline not leaking test information.

### Caveat
This used a **random split**. Near-identical variants of the same malware family can land in both train and test, which inflates scores. Setting `TIME_SPLIT = True` (train on older samples, test on newer) is a harsher and more realistic test, and scores will likely be lower.

## 9. Possible next steps

- **Choose by missed malware, not only F1.** A false negative is usually worse than a false alarm in malware detection. Lowering the decision threshold of the k = 1000 model trades a few extra false alarms for higher recall, at no training cost.
- **Extend the grid.** The best `C` (0.01) sits at the edge of the grid, so try 0.001 and 0.003. The best `k` (1000) is also near the top, so try 1500 or 2000.
- **Run with `TIME_SPLIT = True`** for a more realistic estimate.
- **Add LightGBM** as a reference to see how much room remains above 0.988.